# 04 — Regional clustering

## Pertanyaan

Apakah provinsi dapat disegmentasikan berdasarkan sedikitnya dua indikator BPS resmi pada tahun yang sama?

## Metode

Notebook memilih tahun terbaru dengan minimal dua indikator lengkap pada minimal empat provinsi. Fitur distandardisasi, jumlah cluster 2–6 dipilih dengan silhouette score tertinggi, dan PCA dua dimensi hanya dipakai untuk visualisasi. Missing value tidak diimputasi dan data sintetis tidak dibuat.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from analytics.descriptive.clustering import InsufficientRegionalData, cluster_regional_features, prepare_regional_features
from analytics.descriptive.data_access import DATASET_SOURCES
from analytics.descriptive.notebook_support import insight, prepare_notebook, save_figure

data = prepare_notebook()

## Hasil

In [ ]:
try:
    selected_year, regional_features = prepare_regional_features(data.regional)
    clustering = cluster_regional_features(selected_year, regional_features)
    region_names = (
        data.regional.loc[data.regional["observation_year"].eq(selected_year), ["region_code", "region_name"]]
        .drop_duplicates("region_code")
    )
    assignments = clustering.assignments.merge(region_names, on="region_code", how="left", validate="one_to_one")
    display(regional_features)
    display(assignments)

    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(assignments["pca_1"], assignments["pca_2"], c=assignments["cluster"], cmap="tab10")
    for row in assignments.itertuples():
        ax.annotate(row.region_code, (row.pca_1, row.pca_2), fontsize=8)
    ax.set(title=f"Segmentasi provinsi BPS {selected_year}", xlabel="PCA 1", ylabel="PCA 2")
    fig.colorbar(scatter, ax=ax, label="Cluster")
    save_figure(fig, "04_regional_clustering.png")
    display(fig)
    plt.close(fig)
    selected_rows = data.regional.loc[data.regional["observation_year"].eq(selected_year)]
    display(insight(
        f"{len(assignments)} provinsi terbagi menjadi {clustering.cluster_count} cluster dengan silhouette score {clustering.silhouette:.3f}.",
        frame=selected_rows,
        source=DATASET_SOURCES["regional"],
        limitation="Label cluster bersifat deskriptif dan dapat berubah ketika indikator, periode, atau coverage berubah. PCA hanya proyeksi visual.",
    ))
except InsufficientRegionalData as error:
    display(insight(
        f"Clustering belum dijalankan: {error}.",
        frame=data.regional,
        source=DATASET_SOURCES["regional"],
        limitation="Hasil segmentasi ditunda sampai data produksi BPS memenuhi coverage minimum; tidak ada imputasi atau fallback sintetis.",
    ))

## Interpretasi

Cluster, bila tersedia, menunjukkan provinsi dengan profil indikator yang mirip setelah standardisasi. Nomor cluster tidak menyatakan peringkat baik atau buruk.

## Keterbatasan

Segmentasi memerlukan coverage resmi yang cukup. Silhouette score menilai pemisahan internal, bukan validitas ekonomi; interpretasi akhir harus kembali ke definisi indikator BPS.